# 01b. Parallel Preprocessing via dsub (BAM → PAT → beta)

This notebook submits large-scale BAM→PAT conversion jobs to Google Batch via `dsub`,  
enabling parallel processing of hundreds of samples simultaneously.

**Use this notebook when:**
- Processing large cohorts (>10 samples) where sequential processing would take days/weeks
- You want to fan out across the full All of Us dataset

**For small-scale / testing use `01_preprocess_bam2pat.ipynb` instead.**

## Cohorts Covered
| Cohort | Platform | Site | Samples | BAM size |
|--------|----------|------|---------|----------|
| JHU ONT | Oxford Nanopore | JHU | ~128 | 114–126 GB |
| BCM ONT | Oxford Nanopore | BCM | ~196 | 120–175 GB |
| UW Revio | PacBio Revio | UW | varies | 30–80 GB |
| Broad Revio | PacBio Revio | Broad | varies | 30–80 GB |
| Broad Sequel IIe | PacBio Sequel IIe | Broad | varies | 20–60 GB |

## Required GCS Resources (one-time setup)
Before first use, stage these resources in your bucket:
- `$BUCKET/resources/wgbstools_package.tar.gz` — compiled wgbstools binary
- `$BUCKET/resources/hg38_bundle.tar.gz` — hg38 genome FASTA + index
- `$BUCKET/resources/wgbs_wheels/` — Python wheel directory (pandas, numpy, scipy, pysam)
- `$BUCKET/resources/apt_debs/` — samtools .deb packages (offline install)

## Modes
- `MODE = "TEST"` — 3 samples, stable (non-preemptible) VM, waits for completion
- `MODE = "PRODUCTION"` — all samples, preemptible VM, submits asynchronously

**Estimated cost:** ~\$0.11–0.38/sample/hr × ~100 min/sample

## Cell 1 — Imports and dsub Installation

In [ ]:
import subprocess, sys, os, time
import pandas as pd
import numpy as np
from datetime import datetime

PROJECT_ID = os.environ['GOOGLE_PROJECT']
BUCKET     = os.environ['WORKSPACE_BUCKET']
REGION     = 'us-central1'

# Install dsub
print('Installing dsub...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'dsub', '-q'],
    capture_output=True, text=True
)
if r.returncode == 0:
    ver = subprocess.run('dsub --version', shell=True, capture_output=True, text=True)
    print(f'dsub ready — {ver.stdout.strip() or ver.stderr.strip()}')
else:
    print(f'dsub install failed: {r.stderr[-500:]}')

print(f'\nProject: {PROJECT_ID}')
print(f'Bucket : {BUCKET}')

## Cell 2 — Cohort Configuration

Select the cohort to process by setting `COHORT` and `MODE`.

In [ ]:
# ════════════════════════════════════════════════════════════════
# SELECT COHORT AND MODE
# ════════════════════════════════════════════════════════════════

COHORT = 'JHU_ONT'    # ← Options: 'JHU_ONT', 'BCM_ONT', 'UW_REVIO', 'BROAD_REVIO', 'BROAD_SEQUEL'
MODE   = 'TEST'       # ← 'TEST' (3 samples, stable VM) or 'PRODUCTION' (all, preemptible)

# ── Cohort registry ───────────────────────────────────────────────
COHORT_CONFIGS = {
    'JHU_ONT': {
        'bam_base':      'gs://fc-aou-datasets-controlled/pooled/longreads/v8_delta/JHU/ont/bam',
        'bam_pattern':   '{sample_id}/GRCh38/{sample_id}.bam',
        'disk_gb':       200,
        'machine_type':  'n2-standard-8',
        'minutes_est':   100,
        'platform':      'ONT',
        'description':   'JHU ONT v8_delta (~128 samples, 114-126 GB BAMs)',
    },
    'BCM_ONT': {
        'bam_base':      'gs://fc-aou-datasets-controlled/pooled/longreads/v8_delta/BCM/ont/bam',
        'bam_pattern':   '{sample_id}/GRCh38/{sample_id}.bam',
        'disk_gb':       250,
        'machine_type':  'n2-standard-8',
        'minutes_est':   120,
        'platform':      'ONT',
        'description':   'BCM ONT v8_delta (~196 samples, 120-175 GB BAMs)',
    },
    'UW_REVIO': {
        'bam_base':      'gs://fc-aou-datasets-controlled/pooled/longreads/v8_delta/UW/revio/bam',
        'bam_pattern':   '{sample_id}/GRCh38/{sample_id}.bam',
        'disk_gb':       150,
        'machine_type':  'n2-standard-8',
        'minutes_est':   60,
        'platform':      'Revio',
        'description':   'UW PacBio Revio v8_delta',
    },
    'BROAD_REVIO': {
        'bam_base':      'gs://fc-aou-datasets-controlled/pooled/longreads/v8_delta/Broad/revio/bam',
        'bam_pattern':   '{sample_id}/GRCh38/{sample_id}.bam',
        'disk_gb':       150,
        'machine_type':  'n2-standard-8',
        'minutes_est':   60,
        'platform':      'Revio',
        'description':   'Broad PacBio Revio v8_delta',
    },
    'BROAD_SEQUEL': {
        'bam_base':      'gs://fc-aou-datasets-controlled/pooled/longreads/v8_delta/Broad/sequel2e/bam',
        'bam_pattern':   '{sample_id}/GRCh38/{sample_id}.bam',
        'disk_gb':       120,
        'machine_type':  'n2-standard-8',
        'minutes_est':   50,
        'platform':      'Sequel2e',
        'description':   'Broad PacBio Sequel IIe v8_delta',
    },
}

MODE_SETTINGS = {
    'TEST': {
        'n_samples':   3,
        'preemptible': False,
        'wait':        True,
    },
    'PRODUCTION': {
        'n_samples':   None,  # all
        'preemptible': True,
        'wait':        False,
    },
}

# ── Derived config ────────────────────────────────────────────────
cc  = COHORT_CONFIGS[COHORT]
cfg = MODE_SETTINGS[MODE]

OUT_FOLDER = f'dsub_{COHORT}_{MODE.lower()}' if MODE == 'TEST' else f'dsub_{COHORT}'
OUT_PATH   = f'{BUCKET}/results/{OUT_FOLDER}'
TSV_FILE   = f'dsub_jobs_{COHORT}_{MODE.lower()}.tsv'
SCRIPT_FILE  = f'wgbs_{COHORT.lower()}_job.sh'
CONFIG_FILE  = 'dsub_config.txt'

# GCS resource paths
WGBS_TOOLS_TAR = f'{BUCKET}/resources/wgbstools_package.tar.gz'
HG38_TAR       = f'{BUCKET}/resources/hg38_bundle.tar.gz'
WHEELS_GCS     = f'{BUCKET}/resources/wgbs_wheels'
DEBS_GCS       = f'{BUCKET}/resources/apt_debs'

_n    = cfg['n_samples'] or '(all)'
_rate = 0.38 * (0.30 if cfg['preemptible'] else 1.0)
_n_est = cfg['n_samples'] or 100  # estimate

print(f'{"━"*55}')
print(f'  COHORT:      {COHORT}')
print(f'  Description: {cc["description"]}')
print(f'  MODE:        {MODE}')
print(f'  Samples:     {_n}')
print(f'  Disk/VM:     {cc["disk_gb"]}GB')
print(f'  Machine:     {cc["machine_type"]}')
print(f'  Platform:    {cc["platform"]}')
print(f'  Preemptible: {cfg["preemptible"]}')
print(f'  Output:      {OUT_PATH}/')
print(f'  TSV:         {TSV_FILE}')
print(f'  ~min/sample: {cc["minutes_est"]} min')
print(f'  Est. cost:   ~${_n_est * (cc["minutes_est"]/60) * _rate:.0f} (rough estimate)')
print(f'{"━"*55}')

## Cell 3 — Inspect Cohort Structure

In [ ]:
def gcs_ls(path, requester_pays=True):
    cmd = ['gsutil']
    if requester_pays:
        cmd += ['-u', PROJECT_ID]
    cmd += ['ls', path]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  gsutil ls failed: {r.stderr.strip()[:200]}')
        return []
    return [l.strip() for l in r.stdout.strip().split('\n') if l.strip()]

bam_base = cc['bam_base']
print(f'Scanning: {bam_base}/')
sample_dirs = gcs_ls(f'{bam_base}/')
sample_ids  = [d.rstrip('/').split('/')[-1] for d in sample_dirs if d.strip()]
print(f'Found {len(sample_ids)} sample folders')
print(f'First 5: {sample_ids[:5]}')

## Cell 4 — Discover VPC Network Settings

dsub requires the All of Us network configuration to avoid public IPs.

In [ ]:
print('Discovering VPC network settings...\n')

METADATA = 'http://metadata.google.internal/computeMetadata/v1'
HEADER   = '-H "Metadata-Flavor: Google"'

def curl_meta(path):
    r = subprocess.run(
        f'curl -sf {HEADER} "{METADATA}/{path}"',
        shell=True, capture_output=True, text=True
    )
    return r.stdout.strip() if r.returncode == 0 else None

PET_SA = subprocess.run(
    "gcloud config list account --format='value(core.account)'",
    shell=True, capture_output=True, text=True
).stdout.strip()
assert 'iam.gserviceaccount.com' in PET_SA, f'Unexpected identity: {PET_SA}'

network_uri    = curl_meta('instance/network-interfaces/0/network')
subnetwork_uri = curl_meta('instance/network-interfaces/0/subnetwork')

def short_net(uri):
    if uri and 'projects/' in uri:
        parts = uri.split('/')
        try:
            proj  = parts[parts.index('projects') + 1]
            kind  = parts[-2]  # 'networks' or 'subnetworks'
            name  = parts[-1]
            return f'projects/{proj}/{kind}/{name}'
        except Exception:
            pass
    return uri or 'UNKNOWN'

NETWORK_ARG    = short_net(network_uri)
SUBNETWORK_ARG = short_net(subnetwork_uri)

print(f'Service account : {PET_SA}')
print(f'Network         : {NETWORK_ARG}')
print(f'Subnetwork      : {SUBNETWORK_ARG}')

with open(CONFIG_FILE, 'w') as f:
    f.write(f'PET_SA={PET_SA}\n')
    f.write(f'NETWORK={NETWORK_ARG}\n')
    f.write(f'SUBNETWORK={SUBNETWORK_ARG}\n')
print(f'\nSaved → {CONFIG_FILE}')

## Cell 5 — Build Sample Cohort DataFrame

In [ ]:
def build_cohort_df(bam_base, bam_pattern, sample_ids):
    rows = []
    for sid in sample_ids:
        bam_path = f'{bam_base}/{bam_pattern.format(sample_id=sid)}'
        rows.append({'person_id': sid, 'bam_path': bam_path, 'platform': cc['platform']})
    return pd.DataFrame(rows)

TARGET_DF = build_cohort_df(cc['bam_base'], cc['bam_pattern'], sample_ids)
print(f'Cohort DataFrame: {len(TARGET_DF)} samples')
print(TARGET_DF.head())

## Cell 6 — Verify GCS Resource Cache

In [ ]:
def gcs_pkg_exists(gcs_prefix, pkg):
    r = subprocess.run(
        f"gsutil ls '{gcs_prefix}/{pkg}*' 2>/dev/null | wc -l",
        shell=True, capture_output=True, text=True
    )
    return int(r.stdout.strip() or 0) > 0

REQUIRED_WHEELS = ['pandas', 'numpy', 'scipy', 'pysam']
REQUIRED_DEBS   = ['libdeflate0', 'libhtscodecs2', 'libhts3', 'libncurses6', 'samtools', 'tabix']

print('=== Checking GCS resource cache ===\n')
all_ok = True

def check_res(label, cmd):
    global all_ok
    r  = subprocess.run(cmd, shell=True, capture_output=True)
    ok = r.returncode == 0
    all_ok = all_ok and ok
    print(f'  {"" if ok else ""} {label}')

check_res('wgbstools tar.gz',   f'gsutil -q stat {WGBS_TOOLS_TAR}')
check_res('hg38 bundle tar.gz', f'gsutil -q stat {HG38_TAR}')

print('  Python wheels:')
for pkg in REQUIRED_WHEELS:
    ok = gcs_pkg_exists(WHEELS_GCS, pkg)
    all_ok = all_ok and ok
    print(f'    {"" if ok else ""} {pkg}')

print('  Debian packages:')
for pkg in REQUIRED_DEBS:
    ok = gcs_pkg_exists(DEBS_GCS, pkg)
    all_ok = all_ok and ok
    print(f'    {"" if ok else ""} {pkg}')

print(f'\n{"All resources present" if all_ok else "MISSING RESOURCES — stage them before submitting"}')

## Cell 7 — Write Bash Job Script

This script runs on each dsub VM:  
install Python deps offline → install samtools → unpack wgbstools → download BAM → bam2pat → upload results

In [ ]:
job_script = f"""#!/bin/bash
set -euo pipefail

echo "========================================"
echo " SAMPLE  : ${{SAMPLE_ID}}"
echo " PLATFORM: ${{PLATFORM}}"
echo " HOST    : $(hostname)"
echo " START   : $(date -u)"
echo "========================================"

WORK_DIR="/tmp/wgbs_${{SAMPLE_ID}}"
mkdir -p "$WORK_DIR"
cd "$WORK_DIR"

# ── 0a. Python wheels (offline) ───────────────────────────────────
echo "[0a/6] Installing Python dependencies..."
mkdir -p wheels
gsutil -m -q cp "{WHEELS_GCS}/*.whl" ./wheels/ 2>/dev/null || true
# Also try version-matched subfolders
for py_ver in 310 311 312 313; do
    gsutil -m -q cp "{WHEELS_GCS}/${{py_ver}}/*.whl" ./wheels/ 2>/dev/null || true
done

pip install --no-index --find-links ./wheels \\
    pandas numpy scipy pysam \\
    --quiet --break-system-packages --root-user-action=ignore
echo "       pandas $(python3 -c 'import pandas; print(pandas.__version__)')"

# ── 0b. samtools via .deb (offline) ──────────────────────────────
echo "[0b/6] Installing samtools + htslib..."
mkdir -p debs
gsutil -m -q cp "{DEBS_GCS}/*.deb" ./debs/ 2>/dev/null || true

# Install in dependency order
dpkg -i ./debs/libdeflate0*.deb 2>/dev/null || true
dpkg -i ./debs/libhtscodecs2*.deb 2>/dev/null || true
dpkg -i ./debs/libncurses6*.deb 2>/dev/null || true
dpkg -i ./debs/libhts3*.deb 2>/dev/null || true
dpkg -i ./debs/samtools*.deb ./debs/tabix*.deb 2>/dev/null || true

for tool in samtools bgzip tabix; do
    command -v "$tool" &>/dev/null \\
        && echo "       $tool OK" \\
        || {{ echo "       ERROR: $tool missing"; exit 1; }}
done

# ── 1. wgbstools ─────────────────────────────────────────────────
echo "[1/6] Fetching wgbstools..."
gsutil -q cp {WGBS_TOOLS_TAR} .
tar -xzf wgbstools_package.tar.gz
TOOL="$WORK_DIR/wgbs_tools/wgbstools"
chmod +x "$TOOL"

# ── 2. hg38 genome bundle ────────────────────────────────────────
echo "[2/6] Fetching hg38 genome (~850MB)..."
gsutil -q cp {HG38_TAR} .
tar -xzf hg38_bundle.tar.gz

# ── 3. Initialize genome ─────────────────────────────────────────
echo "[3/6] Initializing hg38 genome..."
"$TOOL" init_genome hg38 -f

# ── 4. Download BAM (requester-pays) ─────────────────────────────
echo "[4/6] Downloading BAM (may take 15-60 min)..."
gcloud storage cp "${{BAM_PATH}}" "${{SAMPLE_ID}}.bam" \\
    --billing-project={PROJECT_ID}
echo "      Size: $(du -sh ${{SAMPLE_ID}}.bam | cut -f1)"

# ── 5. bam2pat ───────────────────────────────────────────────────
echo "[5/6] Running bam2pat..."
"$TOOL" bam2pat "${{SAMPLE_ID}}.bam" --genome hg38 -f

[ -f "${{SAMPLE_ID}}.pat.gz" ] \\
    || {{ echo "ERROR: pat.gz not produced"; exit 1; }}
echo "      pat.gz: $(du -sh ${{SAMPLE_ID}}.pat.gz | cut -f1)"

# ── 6. Upload results ────────────────────────────────────────────
echo "[6/6] Uploading → ${{OUT_PATH}}/"
gcloud storage cp "${{SAMPLE_ID}}.pat.gz" "${{OUT_PATH}}/"
[ -f "${{SAMPLE_ID}}.beta" ] && \\
    gcloud storage cp "${{SAMPLE_ID}}.beta" "${{OUT_PATH}}/"
[ -f "${{SAMPLE_ID}}.pat.gz.tbi" ] && \\
    gcloud storage cp "${{SAMPLE_ID}}.pat.gz.tbi" "${{OUT_PATH}}/"

echo "========================================"
echo " DONE : ${{SAMPLE_ID}}"
echo " END  : $(date -u)"
echo "========================================"
"""

with open(SCRIPT_FILE, 'w') as fh:
    fh.write(job_script)
os.chmod(SCRIPT_FILE, 0o755)
print(f'Job script written: {SCRIPT_FILE}')
print(f'Steps: install-deps → bam2pat → upload')

## Cell 8 — Preflight Checks

In [ ]:
print(f'Preflight checks [{COHORT} | {MODE}]\n')
all_ok = True

def check(label, cmd):
    global all_ok
    r  = subprocess.run(cmd, shell=True, capture_output=True)
    ok = r.returncode == 0
    all_ok = all_ok and ok
    print(f'  {"" if ok else ""} {label}')
    return ok

check('wgbstools tar.gz',   f'gsutil -q stat {WGBS_TOOLS_TAR}')
check('hg38 bundle tar.gz', f'gsutil -q stat {HG38_TAR}')
check('dsub config file',   f'test -f {CONFIG_FILE}')
check('job script file',    f'test -f {SCRIPT_FILE}')
check(f'cohort has samples ({len(TARGET_DF)})', f'test {len(TARGET_DF)} -gt 0')

print('  Python wheels:')
for pkg in REQUIRED_WHEELS:
    ok = gcs_pkg_exists(WHEELS_GCS, pkg)
    all_ok = all_ok and ok
    print(f'    {"" if ok else ""} {pkg}')

print(f'\n{"ALL CHECKS PASSED — safe to submit" if all_ok else "FIX ISSUES ABOVE before submitting"}')

## Cell 9 — Build Job TSV

Checks which samples are already done (skip) and creates the dsub input TSV.

In [ ]:
def is_done(pid: str) -> bool:
    r = subprocess.run(
        f'gsutil -q ls {OUT_PATH}/{pid}.pat.gz',
        shell=True, capture_output=True
    )
    return r.returncode == 0


print(f'Building [{MODE}] TSV — {COHORT}')
print(f'Checking done samples in: {OUT_PATH}/\n')

pool = TARGET_DF.head(cfg['n_samples']) if cfg['n_samples'] else TARGET_DF.sample(frac=1, random_state=42)

queue, skipped = [], 0
for _, row in pool.iterrows():
    pid = str(row['person_id'])
    if is_done(pid):
        print(f'  already done: {pid}')
        skipped += 1
        continue
    queue.append({
        '--env SAMPLE_ID': pid,
        '--env BAM_PATH':  row['bam_path'],
        '--env OUT_PATH':  OUT_PATH,
        '--env PLATFORM':  row['platform'],
    })

if not queue:
    print('\nQueue empty — all samples already done!')
else:
    tsv_df = pd.DataFrame(queue)
    tsv_df.to_csv(TSV_FILE, sep='\t', index=False)

    rate     = 0.38 * (0.30 if cfg['preemptible'] else 1.0)
    est_cost = len(queue) * (cc['minutes_est'] / 60) * rate

    print(f'{"━"*55}')
    print(f'  Platform:    {COHORT}')
    print(f'  To run:      {len(queue)}')
    print(f'  Skipped:     {skipped} (already done)')
    print(f'  TSV:         {TSV_FILE}')
    print(f'  Disk/VM:     {cc["disk_gb"]}GB')
    print(f'  ~min/sample: {cc["minutes_est"]} min')
    print(f'  Est. cost:   ~${est_cost:.0f} total')
    print(f'{"━"*55}')
    print(f'\nFirst 3 rows:')
    print(tsv_df.head(3).to_string(index=False))

## Cell 10 — Submit Jobs

Set `CONFIRMED = True` to submit. Review the printed command first.

In [ ]:
CONFIRMED = False    # ← Set True to submit after reviewing the command below

_net = {}
with open(CONFIG_FILE) as f:
    for line in f:
        k, v = line.strip().split('=', 1)
        _net[k] = v
PET_SA         = _net['PET_SA']
NETWORK_ARG    = _net['NETWORK']
SUBNETWORK_ARG = _net['SUBNETWORK']

LOG_PATH = (
    f'{BUCKET}/logs/{OUT_FOLDER}/'
    f'{datetime.now().strftime("%Y%m%d_%H%M")}/'
)

n_jobs   = len(pd.read_csv(TSV_FILE, sep='\t')) if os.path.exists(TSV_FILE) else 0
rate     = 0.38 * (0.30 if cfg['preemptible'] else 1.0)
est_cost = n_jobs * (cc['minutes_est'] / 60) * rate

flags = [
    'dsub',
    '--provider google-batch',
    f'--project {PROJECT_ID}',
    f'--location {REGION}',
    f'--service-account {PET_SA}',
    '--use-private-address',
    f'--network {NETWORK_ARG}',
    f'--subnetwork {SUBNETWORK_ARG}',
    f'--machine-type {cc["machine_type"]}',
    f'--boot-disk-size {cc["disk_gb"]}',
    '--image google/cloud-sdk:latest',
    f'--script {SCRIPT_FILE}',
    f'--tasks {TSV_FILE}',
    f'--logging {LOG_PATH}',
    f'--name wgbs-{COHORT.lower().replace("_", "-")}',
]
if cfg['preemptible']: flags.append('--preemptible')
if cfg['wait']:        flags.append('--wait')

dsub_cmd = ' \\\n    '.join(flags)

print(f'{"━"*55}')
print(f'  MODE:        {MODE}')
print(f'  Cohort:      {COHORT}')
print(f'  Jobs:        {n_jobs}')
print(f'  Machine:     {cc["machine_type"]} (8 vCPU / 32GB RAM)')
print(f'  Disk:        {cc["disk_gb"]}GB')
print(f'  Preemptible: {cfg["preemptible"]}')
print(f'  Est. cost:   ~${est_cost:.0f}')
print(f'  Logs:        {LOG_PATH}')
print(f'{"━"*55}')
print(f'\nCommand:\n{dsub_cmd}\n')

if not CONFIRMED:
    print('LOCKED — set CONFIRMED = True to submit')
else:
    print(f'Submitting {n_jobs} {COHORT} jobs...')
    start  = time.time()
    result = subprocess.run(dsub_cmd, shell=True, capture_output=True, text=True)
    elapsed = (time.time() - start) / 60
    print(result.stdout)

    if result.returncode == 0:
        print(f'Submitted in {elapsed:.1f} min')
        print(f'Jobs will run ~{cc["minutes_est"]} min each')
        print(f'Monitor with Cell 11')
    else:
        key = [l for l in result.stderr.split('\n')
               if any(kw in l for kw in ['Error', 'denied', 'permission', 'invalid', 'grpc'])]
        print(f'Failed (code {result.returncode})')
        print('\n'.join(key[-8:]))
        print(f'\nLogs: {LOG_PATH}')

## Cell 11 — Monitor Progress

In [ ]:
print(f'{"━"*55}')
print(f'  MONITORING — {COHORT}')
print(f'  Output: {OUT_PATH}/')
print(f'  Time:   {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'{"━"*55}\n')

total = len(TARGET_DF)

r_pat = subprocess.run(
    f'gsutil ls {OUT_PATH}/*.pat.gz 2>/dev/null | wc -l',
    shell=True, capture_output=True, text=True
)
r_beta = subprocess.run(
    f'gsutil ls {OUT_PATH}/*.beta 2>/dev/null | wc -l',
    shell=True, capture_output=True, text=True
)
n_pat  = int(r_pat.stdout.strip()  or 0)
n_beta = int(r_beta.stdout.strip() or 0)
pct    = n_pat / total * 100 if total > 0 else 0

bar = '#' * int(pct / 2)
print(f'  PAT files:  {n_pat:>4} / {total}  [{bar:<50}] {pct:.1f}%')
print(f'  Beta files: {n_beta:>4} / {total}')

if n_pat == total:
    print('\nAll samples complete!')
else:
    print(f'\n{total - n_pat} samples remaining. Re-run this cell to refresh.')

## Cell 12 — Read Failure Logs

Run when Cell 11 shows unexpected failures.

In [ ]:
N_SAMPLES    = 3
SHOW_STDERR  = True
LAST_N_LINES = 40

r = subprocess.run(
    f'gsutil ls {BUCKET}/logs/{OUT_FOLDER}/ 2>/dev/null | sort | tail -1',
    shell=True, capture_output=True, text=True
)
latest = r.stdout.strip()

if not latest:
    print(f'No logs found under {BUCKET}/logs/{OUT_FOLDER}/')
    print('Make sure Cell 10 has been submitted.')
else:
    print(f'Latest log dir: {latest}\n')
    r2 = subprocess.run(f"gsutil ls -r '{latest}'", shell=True, capture_output=True, text=True)
    all_logs = [l.strip() for l in r2.stdout.splitlines() if l.endswith('.log')]

    stderr_logs = [l for l in all_logs if 'stderr' in l]
    sample_logs = stderr_logs[:N_SAMPLES] if SHOW_STDERR else all_logs[:N_SAMPLES]

    for log in sample_logs:
        print(f'\n{"="*60}')
        print(f'Log: {log}')
        print(f'{"="*60}')
        r3 = subprocess.run(f"gsutil cat '{log}' | tail -n {LAST_N_LINES}",
                            shell=True, capture_output=True, text=True)
        print(r3.stdout)